# 2 Gen Reviews

This notebook runs the redesigned review-generation stage for the study.

It generates only original AI NCEMS-criteria reviews for the human proposals.
It does not run novelty reviews, rephrasing, score aggregation, or downstream sampling.

In [ ]:
REVIEW_CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
MODELS_TO_USE = ['gpt-5.5', 'gemini-3.1-pro-preview', 'claude-sonnet-5']

REVIEWS_PER_MODEL_PER_PROPOSAL = 5
GENERATION_TEMPERATURE = 0.9
MAX_TOKENS_REVIEWS = 12000
RETRY_DELAYS = [2, 5, 10]
SAVE_PROGRESS_EVERY_N_CALLS = 10
RESUME_OK = True

# Set to a fixed string only if you want a custom run id.
RUN_ID = None


In [ ]:
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from ai_models_interface import AIModelsInterface
from prompt_templates import PromptManager
from review_generation import (
    build_persona_review_schedule,
    build_review_condition_registry,
    build_review_schedule,
    find_project_root,
    load_human_proposal_roster,
    load_human_review_counts,
    load_reviewer_persona_roster,
    load_shared_call_context,
    run_review_generation_for_condition,
)
from proposal_generation import now_run_id

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
shared_review_context = load_shared_call_context(PROJECT_ROOT)
human_review_target_roster = load_human_proposal_roster(PROJECT_ROOT)
human_review_counts = load_human_review_counts(PROJECT_ROOT)
human_review_target_roster = human_review_target_roster.merge(
    human_review_counts,
    on=['target_cohort', 'target_proposal_id', 'target_proposal_uid'],
    how='left',
    validate='one_to_one',
)
if human_review_target_roster['target_human_n_reviews'].isna().any():
    missing = human_review_target_roster.loc[
        human_review_target_roster['target_human_n_reviews'].isna(),
        ['target_proposal_uid', 'target_proposal_title']
    ]
    raise RuntimeError(f'Missing human review counts for proposals: {missing.to_dict(orient="records")}')

reviewer_persona_roster = load_reviewer_persona_roster(PROJECT_ROOT, human_review_target_roster)
reviewer_persona_schedule = build_persona_review_schedule(
    human_review_target_roster,
    reviewer_persona_roster,
)
review_condition_registry = build_review_condition_registry()
prompt_manager = PromptManager()
ai_interface = AIModelsInterface(config_path='.env', override_env=True)
available_models = ai_interface.get_available_models()
resolved_models = [ai_interface.resolve_model_name(model_name) for model_name in MODELS_TO_USE]
missing_models = [model_name for model_name in resolved_models if model_name not in available_models]
if missing_models:
    raise RuntimeError(f'Requested model(s) unavailable with current API keys: {missing_models}')
if REVIEWS_PER_MODEL_PER_PROPOSAL != 5:
    raise RuntimeError('This redesigned notebook currently assumes 5 reviews per model per proposal.')

stage_run_id = RUN_ID or now_run_id()

print(f'Project root: {PROJECT_ROOT}')
print(f'Human target proposals: {len(human_review_target_roster)}')
print('Human review count distribution:')
print(human_review_target_roster['target_human_n_reviews'].value_counts().sort_index())
print(f'Available canonical models: {available_models}')
print(f'Review conditions to run: {REVIEW_CONDITIONS_TO_RUN}')


In [ ]:
review_generation_outputs = {}
schedule_registry = {}

for condition in REVIEW_CONDITIONS_TO_RUN:
    print(f'\n=== Review Generation: {condition} ===')
    schedule_df = build_review_schedule(
        target_roster=human_review_target_roster,
        condition_config=review_condition_registry[condition],
        models_to_use=resolved_models,
        reviewer_persona_schedule=reviewer_persona_schedule if condition == 'persona' else None,
    )
    schedule_registry[condition] = schedule_df

    condition_result = run_review_generation_for_condition(
        project_root=PROJECT_ROOT,
        ai_interface=ai_interface,
        prompt_manager=prompt_manager,
        shared_review_context=shared_review_context,
        condition_config=review_condition_registry[condition],
        schedule_df=schedule_df,
        generation_temperature=GENERATION_TEMPERATURE,
        max_tokens=MAX_TOKENS_REVIEWS,
        retry_delays=RETRY_DELAYS,
        save_progress_every_n_calls=SAVE_PROGRESS_EVERY_N_CALLS,
        resume_ok=RESUME_OK,
        run_id=stage_run_id,
    )
    review_generation_outputs[condition] = condition_result
    print(f"Schedule file: {condition_result['schedule_path']}")
    print(f"Complete file: {condition_result['complete_path']}")
    print(f"Rows: {len(condition_result['reviews_df'])}")
    if condition_result['qa_issues']:
        print('QA issues:')
        for issue in condition_result['qa_issues'][:10]:
            print(f'  - {issue}')


In [ ]:
summary_rows = []
for condition in REVIEW_CONDITIONS_TO_RUN:
    result = review_generation_outputs[condition]
    reviews_df = result['reviews_df']
    summary_rows.append(
        {
            'condition': condition,
            'schedule_rows': len(schedule_registry[condition]),
            'review_rows': len(reviews_df),
            'target_proposals': reviews_df['target_proposal_uid'].nunique() if not reviews_df.empty else 0,
            'models': ', '.join(sorted(reviews_df['evaluator_model'].dropna().unique())) if not reviews_df.empty else '',
            'distinct_call_ids': reviews_df['review_call_id'].nunique() if not reviews_df.empty else 0,
            'complete_file': str(result['complete_path']) if result['complete_path'] else '',
            'reused_existing': result['reused_existing'],
            'qa_issue_count': len(result['qa_issues']),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df
